# 诗海赫尔墨斯 · Hermes-CNPoetry

[![GitHub](https://img.shields.io/badge/GitHub-CNPoetry--Hermes-2b2620?logo=github)](https://github.com/psknlr/CNPoetry-Hermes)

**中华古典诗词自主规则挖掘与 Skill 生成系统** —— 把 26,720 首古典诗词转化为可回源、可推理、
可比较、可教学、可研究、可调用的规则系统与多智能体。

> 无原文，不成论断。无篇目编号，不成证据。无证据链，不成回答。

本笔记本在 Colab 免费实例上**零依赖**运行全流程（纯 Python 标准库）：

1. 克隆仓库并运行完整流水线（语料 → 计量 → 规则挖掘 → 自主审核 → 归纳 → Skill 编译，约 1–2 分钟）
2. 体验核心能力：检索 / 诗境 / 格律 / 训诂 / 智能体问答 / 多智能体合议 / 创作实验室
3. （可选）在 Colab 里打开中国风 Web 控制台
4. （可选）接入真实大模型（Anthropic / OpenAI 等，经 LiteLLM）

运行方式：菜单 **代码执行程序 → 全部运行**，或逐格 `Shift+Enter`。

研发机构：**医哲未来人工智能研究院（IMPF-AI）**

## 一、克隆仓库 + 运行流水线（零依赖）

In [ ]:
%cd /content
!rm -rf CNPoetry-Hermes
!git clone --depth 1 https://github.com/psknlr/CNPoetry-Hermes.git
%cd CNPoetry-Hermes
!python -m hermes_poetry pipeline --quiet

In [ ]:
# 规则库统计（权威口径）
!python -m hermes_poetry stats

## 二、原文检索与「进入一首诗」诗境

In [ ]:
import json
from hermes_poetry.apps.engine import get_engine

engine = get_engine()

# BM25 + 结构化过滤检索
for h in engine.rag.search("大漠孤烟", top_k=3):
    print(f"《{h['title']}》 {h['author']} · {h['dynasty']}　「{h['quote']}」　{h['poem_id']}")

In [ ]:
# 诗境：逐句 意象/情感标记/平仄/典故候选。
# 注意消歧语法：同题多版本须用 《题名》@作者#首句片段 定解（严格歧义，绝不静默取首）
scene = engine.scene("《春望》")
print(f"《{scene['poem']['title']}》 {scene['poem']['author']}\n")
for ln in scene["lines"]:
    tags = []
    if ln["imagery"]: tags.append("意象:" + "、".join(ln["imagery"]))
    if ln["emotions"]: tags.append("情感:" + "、".join(ln["emotions"]))
    print(f"{ln['line']}　[{ln.get('tone_pattern','—')}]　{'　'.join(tags)}")
print("\n" + scene["note"])

## 三、格律与音韵（广韵 → 平水韵 → 词林正韵）

In [ ]:
from hermes_poetry.extract.phonology import get_phonology

ph = get_phonology()
lines = ["白日依山尽", "黄河入海流", "欲穷千里目", "更上一层楼"]
r = ph.analyze_poem(lines, ["流", "楼"])
print("逐句平仄：")
for ln, pat in zip(lines, r["line_patterns"]):
    print(f"  {ln}　{pat}")
tm = r["template_match"]
print(f"\n标准谱匹配：{tm['best_fit']}（严格位偏差 {tm['deviations']}）")
print(f"首句入韵：{r['first_line_rhymes']}　韵调：{r['rhyme_tone']}")
for f in r["rhyme_feet_phonology"]:
    print(f"  韵脚「{f['char']}」→ 广韵 {f['yun']}｜平水韵 {f['pingshui']}｜词林正韵 {f['cilin']}")
print("\n" + r["note"])

In [ ]:
# 字义训诂（说文解字 + 尔雅，C 层旁证）
g = engine.gloss_query("雎鸠")
for x in g["glosses"]:
    sw = x["shuowen"]["gloss"] if x["shuowen"] else "字书无载"
    print(f"「{x['char']}」 {sw}")

## 四、智能体问答（引用核验 + 论断核验双闸门）

In [ ]:
from hermes_poetry.agent.agent import PoetryAgent

agent = PoetryAgent()
r = agent.ask("明月在古诗里常表达什么情感？")
print(r["answer"])
print("\n引用核验：", "✓ 通过" if r["citation_report"]["ok"] else "✗ 有违规")
print("论断核验：", "✓ 通过" if r["claim_report"]["ok"] else "✗ 有违规")

In [ ]:
# 多智能体合议：规划 → 取证 → 专家 → 批评 → 裁决 → 综合
from hermes_poetry.agent.council import Council

council = Council()
cr = council.deliberate("对比《静夜思》和《月下独酌》的异同")
print(cr["answer"][:1200])
print(f"\n裁决：{cr['decision']}｜置信 {cr['confidence']}")

## 五、创作实验室（今人拟作辅助，永不伪托古人）

In [ ]:
from hermes_poetry.apps.compose import compose_helper, check_draft

helper = compose_helper(genre="七绝", rhyme_char="秋", mood="送别友人", engine=engine)
print(helper["declaration"], "\n")
print("仄起入韵 标准谱：")
for ln in helper["templates"]["仄起入韵"]:
    print(" ", ln)
if "rhyme" in helper:
    print(f"\n「秋」平水韵 {helper['rhyme']['pingshui']}，同部候选字：",
          "".join(helper["rhyme"]["candidates"][:20]))

# 草稿复核：平仄 / 律则 / 撞句
chk = check_draft(["白日依山尽", "黄河入海流", "欲穷千里目", "更上一层楼"])
print(f"\n草稿匹配：{chk['tonal']['template_match']['best_fit']}"
      f"（偏差 {chk['tonal']['template_match']['deviations']}）")
for c in chk["collisions"][:2]:
    print(f"撞句提醒：「{c['line']}」与 {c['with']} 重合「{c['overlaps']}」")

### 词牌全景 · 别名 · 飞花令 · 古风方案（第四轮功能）

In [ ]:
# 新功能速览：词牌全景（龙榆生词谱）/ 诗人字号别名 / 飞花令 / 古风创作方案
r = engine.cipai_query("忆江南")
print("词谱：", r["cipu"]["cipai"], "又名", "、".join(r["cipu"]["aliases"]))
print(r["cipu"]["forms"][0]["pattern"].splitlines()[0], "…（定格首句，○平●仄⊙可平可仄△韵）")
print("语料例词：", len(r["all_poems"]), "首（全部可回源）")

print("\n苏东坡 →", engine.resolve_author("苏东坡"))

f = engine.feihua("花", round_no=1)
print("\n飞花令（令字·花）诗海：", f["reply"]["line"], "——", f["reply"]["title"])
chk = engine.feihua("花", user_line="感时花溅泪")["user_check"]
print("你接「感时花溅泪」→", "✓ 实有" if chk["valid"] else "✗", chk.get("title", ""))

from hermes_poetry.apps.compose import compose_gufeng
g = compose_gufeng(theme="戍边思归", rhyme_char="秋", n_lines=16)
print("\n古风歌行方案：", g["plan"]["segments"], "解换韵；范例",
      [x["title"] for x in g["plan"]["references"]])


## 六、（可选）在 Colab 打开中国风 Web 控制台

16 个功能视图（问诗词/读一首/学诗词/创作/研究端），宣纸昼夜双主题、竖排原文、印章界面。
运行下格后点击输出的链接。

In [ ]:
import subprocess, time
from google.colab import output

proc = subprocess.Popen(["python", "-m", "hermes_poetry", "serve", "--port", "8765"])
time.sleep(8)
output.serve_kernel_port_as_window(8765)  # 弹出新窗口；如被拦截可改用 serve_kernel_port_as_iframe

**公网访问（ngrok 映射）**：上格的 Colab 弹窗仅本人可见；要把界面分享成公开链接，
在 [ngrok 控制台](https://dashboard.ngrok.com/get-started/your-authtoken) 取 authtoken，
存入 Colab 左侧「🔑 密钥」面板（名称 `NGROK_AUTHTOKEN`）后运行下格。

In [ ]:
%pip -q install pyngrok
from google.colab import userdata
from pyngrok import ngrok
ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
tunnel = ngrok.connect(8765, "http")
print("公开链接（任何人可访问，注意关闭）：", tunnel.public_url)
# 结束分享：ngrok.disconnect(tunnel.public_url)；或直接中断本格


## 七、（可选）接入真实大模型（Azure / Poe / MiniMax / LiteLLM）

默认 `local` 后端为确定性演示（离线全功能，可复现）。接入真实模型可解锁
古风歌行 AI 代拟、五层诗义标注器的语篇/诗学层与更强的问答综合——
智能体是模型自主思考选择工具的 ReAct 循环，所有生成内容都要过
引用核验与论断核验双闸门。

下格用表单选择后端：API Key 经 `getpass` 隐藏输入（不落盘不回显）；
MiniMax 支持**国内站**（api.minimaxi.com）与**国际站**（api.minimax.io）切换；
模型名如需更换，配置后执行 `os.environ["HERMES_LLM_MODEL"] = "…"` 即可。

In [ ]:
import os
import getpass

# @markdown ### 选择你要接入的大模型后端
backend = "poe" # @param ["azure", "poe", "minimax", "openai_compat", "litellm"]
# @markdown MiniMax 专用：选择国内站或国际站
minimax_region = "国内" # @param ["国内", "国际"]

print(f"请输入 {backend} 的 API Key:")
api_key = getpass.getpass()

os.environ["HERMES_LLM_BACKEND"] = backend

if backend == "azure":
    os.environ["AZURE_OPENAI_ENDPOINT"] = input("请输入 Azure OpenAI Endpoint 链接: ")
    os.environ["AZURE_OPENAI_API_KEY"] = api_key
    os.environ["HERMES_LLM_MODEL"] = input("请输入部署名 (Deployment Name): ")
elif backend == "poe":
    os.environ["POE_API_KEY"] = api_key
    os.environ["HERMES_LLM_MODEL"] = "Claude-Sonnet-4.6"
elif backend == "minimax":
    os.environ["MINIMAX_API_KEY"] = api_key
    os.environ["MINIMAX_REGION"] = "cn" if minimax_region == "国内" else "intl"
    os.environ["HERMES_LLM_MODEL"] = "MiniMax-M3"
elif backend == "openai_compat":
    os.environ["HERMES_LLM_BASE_URL"] = input("请输入 OpenAI 兼容格式的 Base URL: ")
    os.environ["HERMES_LLM_API_KEY"] = api_key
elif backend == "litellm":
    os.environ["ANTHROPIC_API_KEY"] = api_key
    os.environ["HERMES_LLM_MODEL"] = "anthropic/claude-sonnet-5"
    print("\n[提示] 使用 LiteLLM 需确保已安装：!pip install litellm")

print(f"\n成功配置 {backend} 后端！\n之后重新构造 PoetryAgent()/Council() 即可使用真实大模型。")


---

### 延伸

- 评测：`!python -m hermes_poetry bench`（检索/计量/引用落地三套件 + 六类对抗任务）
- 测试：`!python -m unittest discover -s tests`（86 项，含对抗回归与格律金标）
- 设计文档：[docs/DESIGN.md](https://github.com/psknlr/CNPoetry-Hermes/blob/main/docs/DESIGN.md) ·
  路线图与评审处置：[docs/ROADMAP.md](https://github.com/psknlr/CNPoetry-Hermes/blob/main/docs/ROADMAP.md)
- 数据许可：[THIRD_PARTY_NOTICES.md](https://github.com/psknlr/CNPoetry-Hermes/blob/main/THIRD_PARTY_NOTICES.md)

**架构承继**：Shanghan-Hermes（伤寒-赫尔墨斯）· 证据优先的古典文本智能体范式。